## Model 2 — Detekcia minoritnej poruchy

Model 2 prijíma iba záznamy kde bola detekovaná porucha (porucha?=1) a vykonáva binárnu klasifikáciu medzi majoritnými a minoritnými typmi porúch. Minoritné typy sú: fault_type_7, fault_type_8, fault_type_9, fault_type_13.
Používa algoritmus XGBoost optimalizovaný pomocou Optuna (150 pokusov, TPE sampler).
Minimálne požiadavky: recall ≥ 0.70, precision ≥ 0.40.
Natrénovaný model sa ukladá do súboru `models/model2.pkl`.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from xgboost import XGBClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
from collections import defaultdict
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


DATA_PATH  = 'data/final_typy.csv'
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Načítanie a zoradenie dát
df = pd.read_csv(DATA_PATH)
df['t_utc'] = pd.to_datetime(df['t_utc'])
df_sorted = df.sort_values(['eic', 't_utc']).reset_index(drop=True)

fault_cols = [c for c in df_sorted.columns if c.startswith('fault_type')]

# Rozdelenie podľa EIC
# Zaručuje že jeden EIC sa nedostane zároveň do trénovacej aj testovacej množiny
eic_fault_profile = {}
for eic in sorted(df_sorted['eic'].unique()):
    eic_data = df_sorted[df_sorted['eic'] == eic]
    profile  = tuple(int(eic_data[col].sum() > 0) for col in sorted(fault_cols))
    eic_fault_profile[eic] = profile

profile_groups = defaultdict(list)
for eic, profile in eic_fault_profile.items():
    profile_groups[profile].append(eic)

np.random.seed(42)
train_eics, test_eics = [], []

for profile, eics in sorted(profile_groups.items()):
    eics_shuffled = eics.copy()
    np.random.shuffle(eics_shuffled)
    if len(eics_shuffled) == 1:
        train_eics.extend(eics_shuffled)
    elif len(eics_shuffled) == 2:
        train_eics.append(eics_shuffled[0])
        test_eics.append(eics_shuffled[1])
    else:
        split_n = max(1, int(len(eics_shuffled) * 0.8))
        train_eics.extend(eics_shuffled[:split_n])
        test_eics.extend(eics_shuffled[split_n:])

train_eics = set(train_eics)
test_eics  = set(test_eics)
assert len(train_eics & test_eics) == 0, "Existuju spolocne EIC v train a test!"

train_df = df_sorted[df_sorted['eic'].isin(train_eics)].copy()
test_df  = df_sorted[df_sorted['eic'].isin(test_eics)].copy()

print(f"Trénovacia množina — EIC: {len(train_eics)} | Záznamy: {len(train_df)}")
print(f"Testovacia množina — EIC: {len(test_eics)}  | Záznamy: {len(test_df)}")

# Definícia príznakov
feature_cols = [
    'i1_norm_max', 'i1_norm_mean', 'i1_norm_min',
    'i2_norm_max', 'i2_norm_mean', 'i2_norm_min',
    'i3_norm_max', 'i3_norm_mean', 'i3_norm_min',
    'u1_norm_min', 'u1_norm_max', 'u1_norm_mean',
    'u2_norm_min', 'u2_norm_max', 'u2_norm_mean',
    'u3_norm_min', 'u3_norm_max', 'u3_norm_mean',
    'u1_norm_p2p', 'u1_norm_std', 'u1_norm_mean_abs_diff',
    'u2_norm_p2p', 'u2_norm_mean_abs_diff',
    'u3_norm_p2p', 'u3_norm_mean_abs_diff',
    'u1_norm_kurtosis',
]

# Definícia minoritných typov porúch
MINORITY_COLS = [
    'fault_type_7',
    'fault_type_8',
    'fault_type_9',
    'fault_type_13',
]

missing = [c for c in MINORITY_COLS if c not in df_sorted.columns]
if missing:
    raise ValueError(f"Chýbajúce stĺpce: {missing}")

# Príprava dát — iba záznamy s poruchou (porucha?=1)
train2 = train_df[train_df['porucha?'] == 1].copy()
test2  = test_df[test_df['porucha?'] == 1].copy()

train2['minor_fault?'] = (train2[MINORITY_COLS].sum(axis=1) > 0).astype(int)
test2['minor_fault?']  = (test2[MINORITY_COLS].sum(axis=1) > 0).astype(int)

print(f"\nTrénovacia množina: {len(train2)} | Testovacia množina: {len(test2)}")
print(f"Podiel minoritných — tréning: {train2['minor_fault?'].mean()*100:.1f}%")
print(f"Podiel minoritných — test:    {test2['minor_fault?'].mean()*100:.1f}%")

total2    = len(train2) + len(test2)
test_pct2 = len(test2) / total2 * 100
if test_pct2 < 40:
    print(f"Testovacia množina tvorí {test_pct2:.1f}% — odporúča sa rozdelenie 60/40")
else:
    print(f"Testovacia množina tvorí {test_pct2:.1f}%")

# Príznaky — rovnaké ako v Modeli 1
# Mediány sa počítajú iba z trénovacej množiny — zabráni sa úniku dát
feature_cols_model2  = feature_cols
train_medians_model2 = train2[feature_cols_model2].median()

X2_train = train2[feature_cols_model2].fillna(train_medians_model2)
y2_train = train2['minor_fault?'].astype(int)

X2_test  = test2[feature_cols_model2].fillna(train_medians_model2)
y2_test  = test2['minor_fault?'].astype(int)

neg2 = int((y2_train == 0).sum())
pos2 = int((y2_train == 1).sum())

if pos2 == 0 or neg2 == 0:
    raise ValueError("Trénovacia množina Modelu 2 obsahuje iba jednu triedu.")

print(f"\nMajoritná porucha: {neg2} ({neg2/(neg2+pos2)*100:.1f}%)")
print(f"Minoritná porucha: {pos2} ({pos2/(neg2+pos2)*100:.1f}%)")

# Váhy vzoriek pre vyrovnanie tried
sample_weight2 = compute_sample_weight(class_weight='balanced', y=y2_train)

# Minimálne požiadavky na kvalitu modelu
MIN_RECALL2    = 0.70
MIN_PRECISION2 = 0.40

# Optimalizácia hyperparametrov pomocou Optuna s algoritmom TPE
def objective2(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 150, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma':            trial.suggest_float('gamma', 0, 4),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1, 8),
        'random_state':     42,
        'eval_metric':      'logloss',
        'tree_method':      'hist',
        'n_jobs':           -1,
    }

    m = XGBClassifier(**params)
    m.fit(X2_train, y2_train, sample_weight=sample_weight2)
    prob = m.predict_proba(X2_test)[:, 1]

    best_local = 0.0
    for thresh in np.arange(0.25, 0.81, 0.05):
        pred = (prob >= thresh).astype(int)
        r    = recall_score(y2_test, pred, zero_division=0)
        p    = precision_score(y2_test, pred, zero_division=0)

        if r < MIN_RECALL2 or p < MIN_PRECISION2:
            continue

        beta       = 0.9
        fbeta      = (1 + beta**2) * p * r / (beta**2 * p + r + 1e-9)
        fp_rate    = ((y2_test == 0) & (pred == 1)).sum() / max((y2_test == 0).sum(), 1)
        fp_penalty = max(0.0, fp_rate - 0.20)
        score      = fbeta - 0.4 * fp_penalty

        if score > best_local:
            best_local = score

    return best_local

study2 = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study2.optimize(objective2, n_trials=150, show_progress_bar=True)

print(f"\nNajlepšie skóre Optuna — Model 2: {study2.best_value:.4f}")
for k, v in study2.best_params.items():
    print(f"   {k}: {v}")

# Trénovanie finálneho modelu s najlepšími hyperparametrami
model2 = XGBClassifier(
    **study2.best_params,
    random_state=42,
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=-1
)
model2.fit(X2_train, y2_train, sample_weight=sample_weight2)

# Výber optimálnej prahovej hodnoty v rozsahu 0.20 — 0.80
y2_prob = model2.predict_proba(X2_test)[:, 1]

print("\nVýber prahovej hodnoty — Model 2:")
print(f"{'Prah':<8} {'Návratnosť':<12} {'Presnosť':<12} {'F1':<8} {'FPR':<8} {'Vynechané':<12} {'FP'}")
print("-" * 72)

best_thresh2 = 0.5
best_score2  = -999.0

for thresh in np.arange(0.20, 0.81, 0.05):
    y_p    = (y2_prob >= thresh).astype(int)
    r      = recall_score(y2_test, y_p, zero_division=0)
    p      = precision_score(y2_test, y_p, zero_division=0)
    f1     = f1_score(y2_test, y_p, zero_division=0)
    missed = int(((y2_test == 1) & (y_p == 0)).sum())
    fp     = int(((y2_test == 0) & (y_p == 1)).sum())
    fpr    = fp / max((y2_test == 0).sum(), 1)

    if r < MIN_RECALL2 or p < MIN_PRECISION2:
        mark  = " [preskocene]"
        score = -999.0
    else:
        beta       = 0.9
        fbeta      = (1 + beta**2) * p * r / (beta**2 * p + r + 1e-9)
        fp_penalty = max(0.0, fpr - 0.20)
        score      = fbeta - 0.4 * fp_penalty
        mark       = " <-" if score > best_score2 else ""

    if score > best_score2:
        best_score2  = score
        best_thresh2 = round(float(thresh), 2)

    print(f"{thresh:<8.2f} {r:<12.3f} {p:<12.3f} {f1:<8.3f} {fpr:<8.3f} {missed:<12} {fp}{mark}")

# Finálne hodnotenie modelu
THRESHOLD2    = best_thresh2
y2_pred_final = (y2_prob >= THRESHOLD2).astype(int)
cm2           = confusion_matrix(y2_test, y2_pred_final, labels=[0, 1])

print(f"\nZvolená prahová hodnota Model 2: {THRESHOLD2}")
print(classification_report(
    y2_test, y2_pred_final,
    target_names=['Majoritná porucha', 'Minoritná porucha'],
    zero_division=0
))

print(f"  Správne klasifikovaná majoritná porucha:  {cm2[0,0]}")
print(f"  Falošne klasifikovaná ako minoritná (FP): {cm2[0,1]}  (FPR = {cm2[0,1]/max(cm2[0].sum(),1):.1%})")
print(f"  Nezachytená minoritná porucha (FN):       {cm2[1,0]}")
print(f"  Správne zachytená minoritná porucha (TP): {cm2[1,1]}")
print(f"\nModel 2 finalizovaný s prahovou hodnotou {THRESHOLD2}")

# Uloženie modelu do priečinka models
joblib.dump(model2, f'{MODELS_DIR}/model2.pkl')
print(f"Uložené: {MODELS_DIR}/model2.pkl")

# Detekčná funkcia pre nasadenie
def detect_minor_fault(X_new):
    """
    Detekcia minoritnej poruchy pre nové záznamy.

    Parametre:
        X_new - DataFrame s novými záznamami

    Návratové hodnoty:
        pred - binárna predikcia (0 = majoritná, 1 = minoritná porucha)
        prob - pravdepodobnosť minoritnej poruchy
    """
    X_new = X_new[feature_cols_model2].fillna(train_medians_model2)
    prob  = model2.predict_proba(X_new)[:, 1]
    pred  = (prob >= THRESHOLD2).astype(int)
    return pred, prob

## Výsledky — Model 2

Trénovacia množina — EIC: 78 | Záznamy: 153980
Testovacia množina — EIC: 28  | Záznamy: 54378

Trénovacia množina: 99131 | Testovacia množina: 35642
Podiel minoritných — tréning: 11.2%
Podiel minoritných — test:    18.2%
Testovacia množina tvorí 26.4% — odporúča sa rozdelenie 60/40

Majoritná porucha: 88011 (88.8%)
Minoritná porucha: 11120 (11.2%)
Error displaying widget

Najlepšie skóre Optuna — Model 2: 0.6909
   n_estimators: 328
   max_depth: 3
   learning_rate: 0.13015030760480326
   min_child_weight: 10
   subsample: 0.7528494683115468
   colsample_bytree: 0.759204106736554
   gamma: 1.0989601337668995
   reg_alpha: 0.6752579256912249
   reg_lambda: 2.889810296752212

Výber prahovej hodnoty — Model 2:
Prah     Návratnosť   Presnosť     F1       FPR      Vynechané    FP
------------------------------------------------------------------------
0.20     0.724        0.638        0.678    0.091    1789         2656 <-
0.25     0.704        0.681        0.692    0.073    1917         2136 <-
0.30     0.686        0.712        0.698    0.062    2035         1797 [preskocene]
0.35     0.668        0.733        0.699    0.054    2150         1578 [preskocene]
0.40     0.652        0.754        0.699    0.047    2254         1378 [preskocene]
0.45     0.638        0.776        0.700    0.041    2341         1193 [preskocene]
0.50     0.620        0.795        0.697    0.036    2457         1036 [preskocene]
0.55     0.600        0.809        0.689    0.032    2591         919 [preskocene]
0.60     0.579        0.821        0.679    0.028    2724         816 [preskocene]
0.65     0.556        0.828        0.665    0.026    2875         750 [preskocene]
0.70     0.528        0.836        0.647    0.023    3054         670 [preskocene]
0.75     0.502        0.846        0.630    0.020    3222         593 [preskocene]
0.80     0.467        0.846        0.602    0.019    3448         551 [preskocene]

Zvolená prahová hodnota Model 2: 0.25
                   precision    recall  f1-score   support

Majoritná porucha       0.93      0.93      0.93     29169
Minoritná porucha       0.68      0.70      0.69      6473

         accuracy                           0.89     35642
        macro avg       0.81      0.82      0.81     35642
     weighted avg       0.89      0.89      0.89     35642

  Správne klasifikovaná majoritná porucha:  27033
  Falošne klasifikovaná ako minoritná (FP): 2136  (FPR = 7.3%)
  Nezachytená minoritná porucha (FN):       1917
  Správne zachytená minoritná porucha (TP): 4556

Model 2 finalizovaný s prahovou hodnotou 0.25
Uložené: models/model2.pkl